# spaCy en accion: pipeline completo y material de apoyo

**Duracion estimada:** 35 minutos

## Proposito
Este notebook funciona como material de consolidacion. Reune en un mismo recorrido las etapas principales del pipeline de `spaCy` y las conecta con una aplicacion sencilla de extraccion de palabras clave.

## Relacion con el notebook anterior
- En `001_spacy_fundamentos.ipynb` trabajaste ejemplos breves y focalizados.
- Aca vas a mirar un flujo mas continuo: texto -> procesamiento -> analisis -> visualizacion.


In [ ]:
!pip install spacy -q
!python -m spacy download es_core_news_sm -q


In [ ]:
import spacy
from collections import Counter
from spacy import displacy
from wordcloud import WordCloud
import matplotlib.pyplot as plt

nlp = spacy.load("es_core_news_sm")


## 1. El pipeline en una sola mirada

Cuando procesas un texto con `spaCy`, el modelo aplica una secuencia de operaciones. Ver ese pipeline te ayuda a entender por que despues aparecen tokens, lemas, dependencias y entidades.


In [ ]:
print(nlp.pipe_names)


In [ ]:
texto_ejemplo = "La empresa espanola TechSolutions S.L. abrira una nueva sede en Argentina el proximo ano."
doc = nlp(texto_ejemplo)
print(texto_ejemplo)


## 2. Tokenizacion

Primer paso: observar como el modelo divide el texto.


In [ ]:
tokens = [token.text for token in doc]
print(tokens)


## 3. Lematizacion

Ahora miramos la forma base de cada token, excluyendo puntuacion y espacios para que la salida quede mas clara.


In [ ]:
for token in doc:
    if not token.is_punct and not token.is_space:
        print(f"'{token.text}' -> '{token.lemma_}'")


## 4. Etiquetado gramatical (POS)

Esta salida te permite leer que rol gramatical cumple cada palabra dentro del texto.


In [ ]:
for token in doc:
    if not token.is_space:
        print(f"'{token.text}' -> {token.pos_} ({spacy.explain(token.pos_)}) -> {token.tag_}")


## 5. Dependencias sintacticas

Aca aparece la relacion entre cada token y su palabra cabeza. Esta informacion es muy util para interpretar estructura.


In [ ]:
for token in doc:
    if not token.is_space:
        print(f"'{token.text}' -> {token.dep_} ({spacy.explain(token.dep_)}) -> '{token.head.text}'")


In [ ]:
displacy.render(doc, style='dep', jupyter=True, options={'distance': 120})


## 6. Reconocimiento de entidades nombradas

En este ejemplo, el foco esta puesto en detectar organizaciones y lugares relevantes.


In [ ]:
if doc.ents:
    print("Entidades encontradas:")
    print("Texto de la entidad -> Etiqueta (tipo)")
    for ent in doc.ents:
        print(f"'{ent.text}' -> {ent.label_} ({spacy.explain(ent.label_)})")
else:
    print("No se encontraron entidades nombradas en este texto.")


In [ ]:
displacy.render(doc, style='ent', jupyter=True, options={'distance': 200})


## 7. Ejemplo aplicado: palabras clave en un texto expositivo

Hasta ahora trabajamos con un ejemplo breve. En esta parte vamos a usar un texto mas largo para extraer palabras clave y visualizar sus frecuencias.


In [ ]:
wiki_txt = """
La fotosintesis es el proceso quimico fundamental mediante el cual las plantas verdes,
algas y algunas bacterias convierten la energia luminica del sol en energia quimica.
Utilizan dioxido de carbono del aire y agua del suelo para producir glucosa,
su alimento principal, liberando oxigeno como subproducto. La clorofila,
un pigmento verde en los cloroplastos, es crucial para capturar esta energia solar.
Este proceso sustenta casi toda la vida en la Tierra.
"""

print(wiki_txt[:180] + "...")
doc = nlp(wiki_txt)
print("Texto procesado.")


## 8. Extraccion de palabras clave

En esta estrategia nos quedamos con tokens alfabeticos que no son stopwords y usamos el lema para agrupar variantes.


In [ ]:
palabras_clave = []
for token in doc:
    if token.is_alpha and not token.is_stop:
        palabras_clave.append(token.lemma_.lower())

print(f"Se extrajeron {len(palabras_clave)} palabras clave.")
print("Ejemplo:", palabras_clave[:15])


In [ ]:
frecuencia_palabras = Counter(palabras_clave)
palabras_mas_comunes = frecuencia_palabras.most_common(15)

for palabra, frecuencia in palabras_mas_comunes:
    print(f"- '{palabra}': {frecuencia}")


## 9. Visualizacion

La nube de palabras no reemplaza el analisis, pero puede ayudarte a detectar temas dominantes de manera rapida.


In [ ]:
wordcloud_generator = WordCloud(
    width=800,
    height=400,
    background_color='white',
    colormap='viridis',
    max_words=50,
    collocations=False,
).generate_from_frequencies(frecuencia_palabras)


In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud_generator, interpolation='bilinear')
plt.axis("off")
plt.tight_layout(pad=0)
plt.show()


## Cierre y puente hacia el prelaboratorio

Si queres profundizar este material, proba este ejercicio:
1. Pedile a un asistente de IA que te proponga cinco palabras clave posibles para el texto sobre fotosintesis.
2. Compara esa seleccion con `palabras_mas_comunes`.
3. Explica cuales conservarias, cuales descartarias y por que.

En el proximo notebook vas a aplicar esta logica a una noticia, con un foco mas claro en lectura critica y preparacion para el laboratorio.
